In [21]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ensembles import *
from functions.repeated_cv import *

from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import make_scorer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import integrated_brier_score
from sksurv.metrics import concordance_index_ipcw
from sksurv.nonparametric import kaplan_meier_estimator
from sklearn.inspection import permutation_importance

from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

import time
set_config(display="text")  # displays text representation of estimators

In [22]:
dataset = pd.read_csv("../../data/data_final.csv")

diagnosis_variables = []

with open("../../data/diagnosis_final.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

treatment_variables = []
with open("../../data/treatment_final.txt", "r") as file:
    for line in file:
        treatment_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = treatment_variables + diagnosis_variables

In [23]:
# Find duplicated patients
mask = (survival_dataset.duplicated(subset='PERSON_ID', keep=False)) #& (survival_dataset['Regional LN recurrence'] == 1)

# For each duplicated PERSON_ID with recurrence, keep only the row with the lowest survival days
to_remove = (
    survival_dataset[mask]
    .sort_values(['PERSON_ID', 'Survival days'], ascending=[True, True])
    .groupby('PERSON_ID')
    .head(-1)
    .index
)

survival_dataset = survival_dataset.drop(to_remove)

In [24]:
# Prepare data
data_x = survival_dataset[feature_columns]
data_y = Surv.from_arrays(
    event=survival_dataset['Regional LN recurrence'],
    time=survival_dataset['Survival days']
)

In [25]:
sgkf = StratifiedKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Regional LN recurrence'])


train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

In [26]:
param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
    'rank_ratio': [0.0, 0.5, 1.0],
    'tol': [1e-3, 1e-4],
    'max_iter': [5000],
}
metrics = ['c-index', 'c-index-ipcw']

In [27]:
results_df, summary_df, best_params = ensemble_repeated_grid_search(
    FastSurvivalSVM, x_train, y_train, param_grid, metrics,
                                  model_name='SVM Model',
                                  scale_data=True, scaler_type='standard',
                                  random_state=None, n_jobs=-1,
                                  n_controls_per_case=None, dataset=None,
                                  feature_columns=None,
                                  event_col='Regional LN recurrence',
                                  time_col='Survival days',
                                  n_splits=4, n_repeats=5, base_random_state=42,
                                  verbose=True)


SVM MODEL REPEATED GRID SEARCH
Model: FastSurvivalSVM
Scaling: Yes (standard)
Subsampling: No
Metrics: ['c-index', 'c-index-ipcw']
Parameter Grid:
  - alpha: [0.01, 0.1, 1.0, 10.0, 100.0]
  - rank_ratio: [0.0, 0.5, 1.0]
  - tol: [0.001, 0.0001]
  - max_iter: [5000]
Total hyperparameter combinations: 30
Folds per repetition: 4
Repetitions: 5
Total evaluations: 600

[1/30] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.001, 'max_iter': 5000}
  c-index: 0.5759 ± 0.1540
  c-index-ipcw: 0.5127 ± 0.1942
[2/30] Testing params: {'alpha': 0.01, 'rank_ratio': 0.0, 'tol': 0.0001, 'max_iter': 5000}
  Elapsed: 0.1min, Est. remaining: 3.1min
  c-index: 0.5760 ± 0.1538
  c-index-ipcw: 0.5133 ± 0.1942
[3/30] Testing params: {'alpha': 0.01, 'rank_ratio': 0.5, 'tol': 0.001, 'max_iter': 5000}
  Elapsed: 0.3min, Est. remaining: 4.6min
  c-index: 0.1974 ± 0.0849
  c-index-ipcw: 0.3224 ± 0.2150
[4/30] Testing params: {'alpha': 0.01, 'rank_ratio': 0.5, 'tol': 0.0001, 'max_iter': 5000}
  Elapsed

In [28]:
summary_df.to_csv('../../results/repeated_full/ssvm_full.csv', index=False)